# Mass Maps FIX-first iterative feature improvement

This notebook adapts the earlier ChestX notebook to the **official Mass Maps / CosmoGrid** FIX setting.

## Important note

The Hugging Face dataset you linked is **mass maps cosmology**, not mass spectrometry.

In the FIX benchmark, the Mass Maps task:
- uses **one-channel 66x66 weak-lensing maps**
- predicts cosmological parameters like **Omega_m** and **sigma_8**
- evaluates interpretability with an **implicit** expert-alignment score based on **void/cluster purity**, not explicit segmentation masks

That means the optimization setup is different from ChestX:
- **ChestX**: explicit expert structures, compare with anatomical masks
- **Mass Maps**: implicit expert features, score groups by how purely they capture **voids** or **clusters**

So here the right question is:

> Can iterative feature improvement increase **MassMapsFixScore**?

This notebook answers that in three stages:
1. **FIX-only search**
2. **FIX-only search with restarts / beam search**
3. **Actual REINFORCE-style action policy**

In [ ]:
# Optional install for a fresh runtime
# %pip install -q exlib matplotlib scikit-image pandas tqdm

In [ ]:
import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from skimage import measure, morphology
from tqdm.auto import tqdm

from exlib.datasets.mass_maps import MassMapsDataset, MassMapsFixScore
from exlib.features.vision.watershed import WatershedGroups
from exlib.features.vision.quickshift import QuickshiftGroups
from exlib.features.vision.patch import PatchGroups
from exlib.features.vision.mass_maps import MassMapsOracle, MassMapsOne

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Load the official dataset and FIX scorer

The FIX paper describes the Mass Maps dataset as:
- clean simulations from CosmoGridV1
- input shape `(1, 66, 66)`
- target cosmology parameters
- expert alignment based on **voids** and **clusters**

This is an **implicit** expert-alignment setting.

In [ ]:
train_dataset = MassMapsDataset(split="train")
val_dataset = MassMapsDataset(split="validation")
test_dataset = MassMapsDataset(split="test")

print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))

massmaps_fix = MassMapsFixScore().to(device)
print("MassMapsFixScore ready.")

## Helpers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_numpy01(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().float().numpy()
    x = np.asarray(x, dtype=np.float32).squeeze()
    if x.size == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if hi - lo < 1e-8:
        return np.zeros_like(x, dtype=np.float32)
    return (x - lo) / (hi - lo)


def mask_iou(a, b) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / max(float(union), 1.0)


def dedup_masks(masks, iou_thr=0.70):
    kept = []
    for mask in masks:
        mask = mask.astype(bool)
        if mask.sum() == 0:
            continue
        if all(mask_iou(mask, prev) < iou_thr for prev in kept):
            kept.append(mask)
    return kept


def groups_to_tensor(groups, device):
    arr = np.stack([g.astype(np.float32) for g in groups], axis=0)
    return torch.from_numpy(arr).unsqueeze(0).to(device)


def connected_components(mask, min_area=20):
    labels = measure.label(mask)
    comps = []
    for region in sorted(measure.regionprops(labels), key=lambda r: r.area, reverse=True):
        if region.area < min_area:
            continue
        comp = labels == region.label
        comp = morphology.remove_small_holes(comp, area_threshold=max(16, min_area))
        comp = morphology.remove_small_objects(comp, min_size=min_area)
        if comp.sum() >= min_area:
            comps.append(comp.astype(bool))
    return comps

In [ ]:
@torch.no_grad()
def score_groups_massmaps(fix_scorer, image_tensor, groups):
    groups_t = groups_to_tensor(groups, device=next(fix_scorer.parameters()).device if any(True for _ in fix_scorer.parameters()) else device)
    x_t = image_tensor.to(groups_t.device)

    try:
        out = fix_scorer(groups_t, x_t, reduce='none', return_dict=True)
        # expected keys in official notebook:
        # p_void_, p_cluster_, purity, maybe score / alignment
        fix = None
        for key in ["score", "scores", "alignment", "fix", "fixscore"]:
            if key in out:
                val = out[key]
                fix = float(val.flatten()[0].item())
                break
        if fix is None:
            # fallback: if the scorer returns only component scores, combine purity-like summary
            if "purity" in out:
                fix = float(out["purity"].mean().item())
            else:
                raise KeyError(f"Could not find a score key in scorer output keys={list(out.keys())}")
        metrics = {"fix": fix}
        for k in ["p_void_", "p_cluster_", "purity", "ratio_vc", "ratio"]:
            if k in out:
                metrics[k] = float(out[k].flatten().mean().item())
        return metrics
    except TypeError:
        # fallback in case return_dict is unsupported
        out = fix_scorer(groups_t, x_t)
        return {"fix": float(out.flatten()[0].item())}


def diversity_score(groups):
    if len(groups) <= 1:
        return 1.0
    vals = []
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            vals.append(mask_iou(groups[i], groups[j]))
    return 1.0 - float(np.mean(vals))


def metrics_fix_only(fix_scorer, image_tensor, groups):
    out = score_groups_massmaps(fix_scorer, image_tensor, groups)
    out["div"] = diversity_score(groups)
    out["num_groups"] = len(groups)
    return out


def better_by_fix_then_div(a: Dict[str, float], b: Dict[str, float], eps: float = 1e-8) -> bool:
    if a["fix"] > b["fix"] + eps:
        return True
    if abs(a["fix"] - b["fix"]) <= eps and a["div"] > b["div"] + eps:
        return True
    return False

## Build a candidate bank

Unlike ChestX, this notebook does **not** rely on explicit structure masks.

Instead, it builds a proposal bank from segmentation-style group generators:
- watershed
- quickshift
- patch
- optional mass-maps-specific helpers (`MassMapsOne`, `MassMapsOracle`) if available

This is a better fit for the implicit Mass Maps expert alignment.

In [ ]:
watershed_groups = WatershedGroups(min_dist=10, compactness=0).to(device)
quickshift_groups = QuickshiftGroups().to(device)
patch_groups = PatchGroups(grid_size=(8, 8), mode='grid').to(device)

try:
    massmaps_one = MassMapsOne().to(device)
except Exception:
    massmaps_one = None

try:
    massmaps_oracle = MassMapsOracle().to(device)
except Exception:
    massmaps_oracle = None

print("baseline group generators ready.")

In [ ]:
def tensor_groups_to_masks(groups_tensor, min_area=20):
    groups_np = groups_tensor.detach().cpu().numpy()
    if groups_np.ndim == 4:
        groups_np = groups_np[0]
    masks = []
    for g in groups_np:
        mask = g > 0
        if mask.sum() < min_area:
            continue
        comps = connected_components(mask, min_area=min_area)
        if comps:
            masks.extend(comps)
        else:
            masks.append(mask.astype(bool))
    return masks


def build_candidate_bank_massmaps(image_tensor, max_candidates=32, min_area=20):
    proposals = []
    provenance = []

    generators = [
        ("watershed", watershed_groups),
        ("quickshift", quickshift_groups),
        ("patch", patch_groups),
    ]
    if massmaps_one is not None:
        generators.append(("massmaps_one", massmaps_one))
    if massmaps_oracle is not None:
        generators.append(("massmaps_oracle", massmaps_oracle))

    for name, generator in generators:
        try:
            groups = generator(image_tensor.to(device))
            masks = tensor_groups_to_masks(groups, min_area=min_area)
            proposals.extend(masks)
            provenance.extend([name] * len(masks))
        except Exception as e:
            print(f"Skipping {name} due to error: {e}")

    # add simple threshold-based proposals directly from image intensity
    image01 = to_numpy01(image_tensor[0, 0])
    for q in [0.05, 0.10, 0.15, 0.85, 0.90, 0.95]:
        thresh = np.quantile(image01, q)
        if q < 0.5:
            base = image01 <= thresh
        else:
            base = image01 >= thresh
        base = morphology.binary_opening(base, morphology.disk(1))
        base = morphology.binary_closing(base, morphology.disk(2))
        comps = connected_components(base, min_area=min_area)
        proposals.extend(comps)
        provenance.extend([f"quantile_{q:.2f}"] * len(comps))

    proposals = dedup_masks(proposals)[:max_candidates]
    provenance = provenance[:len(proposals)]

    return {
        "groups": proposals,
        "provenance": provenance,
        "image01": image01,
    }

In [ ]:
def ranked_singletons(fix_scorer, image_tensor, candidate_bank):
    rows = []
    for i, g in enumerate(candidate_bank):
        m = metrics_fix_only(fix_scorer, image_tensor, [g])
        rows.append({
            "idx": i,
            "fix": m["fix"],
            "div": m["div"],
            "p_void_": m.get("p_void_", np.nan),
            "p_cluster_": m.get("p_cluster_", np.nan),
            "purity": m.get("purity", np.nan),
        })
    df = pd.DataFrame(rows).sort_values(["fix", "div"], ascending=False).reset_index(drop=True)
    return df


def seed_init_groups(candidate_bank, singleton_df, init_groups=4):
    idxs = singleton_df["idx"].tolist()[:min(init_groups, len(singleton_df))]
    return [candidate_bank[i].copy() for i in idxs]

## Visualization

In [ ]:
def show_overlay(ax, image01, mask, title="", color=(1.0, 0.15, 0.15), alpha=0.35):
    ax.imshow(image01, cmap="viridis")
    rgba = np.zeros((*mask.shape, 4), dtype=np.float32)
    rgba[..., :3] = np.array(color, dtype=np.float32)
    rgba[..., 3] = alpha * mask.astype(np.float32)
    ax.imshow(rgba)
    for contour in measure.find_contours(mask.astype(float), 0.5):
        ax.plot(contour[:, 1], contour[:, 0], color="white", lw=1.0)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


def plot_group_grid(image01, groups, title):
    cols = min(4, max(1, len(groups)))
    rows = int(math.ceil(max(1, len(groups)) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    colors = [
        (1.0, 0.15, 0.15),
        (0.15, 0.85, 1.0),
        (0.35, 1.0, 0.35),
        (1.0, 0.8, 0.1),
        (0.9, 0.3, 1.0),
        (1.0, 0.5, 0.2),
    ]
    for i, ax in enumerate(axes):
        if i < len(groups):
            show_overlay(ax, image01, groups[i], title=f"group {i}", color=colors[i % len(colors)])
        else:
            ax.axis("off")
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()

## FIX-only search baseline

This is the clean diagnostic baseline:
- accept by **FIX**
- track best by **FIX**
- use diversity only as a tiebreaker

This tells you whether MassMapsFixScore can drive iterative improvement at all.

In [ ]:
def mutate_mask(mask, image01, rng):
    op = rng.choice(["dilate", "erode", "open", "close", "fill", "largest", "threshold_grow"])
    k = int(rng.integers(1, 4))
    selem = morphology.disk(k)
    out = mask.copy()

    if op == "dilate":
        out = morphology.binary_dilation(out, selem)
    elif op == "erode":
        out = morphology.binary_erosion(out, selem)
    elif op == "open":
        out = morphology.binary_opening(out, selem)
    elif op == "close":
        out = morphology.binary_closing(out, selem)
    elif op == "fill":
        out = morphology.remove_small_holes(out, area_threshold=36)
    elif op == "largest":
        comps = connected_components(out, min_area=1)
        if comps:
            out = comps[0]
    elif op == "threshold_grow":
        outer = morphology.binary_dilation(out, morphology.disk(2))
        vals = image01[outer]
        if vals.size > 0:
            lo, hi = np.quantile(vals, 0.2), np.quantile(vals, 0.8)
            grow_dark = image01 <= lo
            grow_bright = image01 >= hi
            out = np.logical_or(out, np.logical_and(outer, np.logical_or(grow_dark, grow_bright)))

    out = morphology.remove_small_objects(out.astype(bool), min_size=20)
    out = morphology.remove_small_holes(out.astype(bool), area_threshold=24)
    return out.astype(bool)


def sanitize_groups(groups, candidate_bank, min_area=20):
    proposal = [g for g in dedup_masks(groups) if g.sum() >= min_area]
    if not proposal and len(candidate_bank) > 0:
        proposal = [candidate_bank[0].copy()]
    return proposal


def apply_action(proposal, action, candidate_bank, image01, rng, max_groups=8):
    proposal = [g.copy() for g in proposal]

    if action == "modify" and proposal:
        idx = int(rng.integers(0, len(proposal)))
        proposal[idx] = mutate_mask(proposal[idx], image01, rng)

    elif action == "add" and len(proposal) < max_groups and candidate_bank:
        proposal.append(candidate_bank[int(rng.integers(0, len(candidate_bank)))].copy())

    elif action == "delete" and len(proposal) > 1:
        proposal.pop(int(rng.integers(0, len(proposal))))

    elif action == "replace" and proposal and candidate_bank:
        idx = int(rng.integers(0, len(proposal)))
        proposal[idx] = candidate_bank[int(rng.integers(0, len(candidate_bank)))].copy()

    elif action == "merge" and len(proposal) > 1:
        i, j = sorted(rng.choice(len(proposal), size=2, replace=False).tolist())
        proposal[i] = np.logical_or(proposal[i], proposal[j])
        proposal.pop(j)

    return sanitize_groups(proposal, candidate_bank)

In [ ]:
def improve_groups_fix_only(
    fix_scorer,
    image_tensor,
    init_groups,
    candidate_bank,
    num_steps=120,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    current = [g.copy() for g in init_groups]
    current_metrics = metrics_fix_only(fix_scorer, image_tensor, current)

    best = [g.copy() for g in current]
    best_metrics = dict(current_metrics)

    history = [dict(step=0, accepted=True, action="init", **current_metrics)]

    for step in range(1, num_steps + 1):
        action = rng.choice(["modify", "add", "delete", "replace", "merge"])
        proposal = apply_action(current, action, candidate_bank, image01, rng, max_groups=max_groups)
        proposal_metrics = metrics_fix_only(fix_scorer, image_tensor, proposal)

        accept = better_by_fix_then_div(proposal_metrics, current_metrics)
        if accept:
            current = [g.copy() for g in proposal]
            current_metrics = dict(proposal_metrics)

        if better_by_fix_then_div(current_metrics, best_metrics):
            best = [g.copy() for g in current]
            best_metrics = dict(current_metrics)

        history.append(dict(step=step, accepted=accept, action=action, **current_metrics))

    return best, history, best_metrics

## Stronger search: random restarts and beam search

In [ ]:
def run_fix_search_with_restarts(
    fix_scorer,
    image_tensor,
    candidate_bank,
    n_restarts=8,
    num_steps=80,
    init_groups=4,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    singleton_df = ranked_singletons(fix_scorer, image_tensor, candidate_bank)

    all_runs = []
    best_groups = None
    best_metrics = None

    for r in range(n_restarts):
        if r == 0:
            start_groups = seed_init_groups(candidate_bank, singleton_df, init_groups=init_groups)
        else:
            order = rng.permutation(len(candidate_bank)).tolist()
            chosen = order[:min(init_groups, len(order))]
            start_groups = [candidate_bank[i].copy() for i in chosen]

        groups, history, metrics = improve_groups_fix_only(
            fix_scorer=fix_scorer,
            image_tensor=image_tensor,
            init_groups=start_groups,
            candidate_bank=candidate_bank,
            num_steps=num_steps,
            max_groups=max_groups,
            seed=int(rng.integers(0, 10_000_000)),
        )
        all_runs.append({"restart": r, "groups": groups, "history": history, "metrics": metrics})

        if best_metrics is None or better_by_fix_then_div(metrics, best_metrics):
            best_groups = [g.copy() for g in groups]
            best_metrics = dict(metrics)

    return best_groups, all_runs, best_metrics, singleton_df

In [ ]:
def beam_search_fix(
    fix_scorer,
    image_tensor,
    candidate_bank,
    beam_width=4,
    expansions_per_state=4,
    num_rounds=12,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    singleton_df = ranked_singletons(fix_scorer, image_tensor, candidate_bank)
    beam = []

    for idx in singleton_df["idx"].tolist()[:beam_width]:
        groups = [candidate_bank[idx].copy()]
        metrics = metrics_fix_only(fix_scorer, image_tensor, groups)
        beam.append({"groups": groups, "metrics": metrics, "trace": [f"start:{idx}"]})

    if not beam:
        raise ValueError("Candidate bank is empty.")

    history = []

    for round_idx in range(num_rounds):
        candidates = []
        for state in beam:
            candidates.append(state)
            for _ in range(expansions_per_state):
                action = rng.choice(["modify", "add", "delete", "replace", "merge"])
                proposal = apply_action(state["groups"], action, candidate_bank, image01, rng, max_groups=max_groups)
                metrics = metrics_fix_only(fix_scorer, image_tensor, proposal)
                candidates.append({
                    "groups": [g.copy() for g in proposal],
                    "metrics": metrics,
                    "trace": state["trace"] + [action],
                })

        candidates.sort(key=lambda s: (s["metrics"]["fix"], s["metrics"]["div"]), reverse=True)

        next_beam = []
        for cand in candidates:
            keep = True
            for existing in next_beam:
                if len(cand["groups"]) == len(existing["groups"]):
                    mean_iou = np.mean([
                        max(mask_iou(g, h) for h in existing["groups"])
                        for g in cand["groups"]
                    ])
                    if mean_iou > 0.80:
                        keep = False
                        break
            if keep:
                next_beam.append(cand)
            if len(next_beam) >= beam_width:
                break

        beam = next_beam
        history.append({
            "round": round_idx,
            "beam_fix": [s["metrics"]["fix"] for s in beam],
            "beam_div": [s["metrics"]["div"] for s in beam],
        })

    best = beam[0]
    return best["groups"], history, best["metrics"], best["trace"]

## Actual RL-style policy learning

This is the part that is genuinely RL-like.

It learns a small action policy over:
- modify
- add
- delete
- replace
- merge

Reward:
- `delta FIX = new_fix - old_fix`

In [ ]:
ACTION_NAMES = ["modify", "add", "delete", "replace", "merge"]


def state_features(current_groups, current_metrics, max_groups=8):
    return np.array([
        current_metrics["fix"],
        current_metrics["div"],
        current_metrics.get("p_void_", 0.0),
        current_metrics.get("p_cluster_", 0.0),
        current_metrics.get("purity", 0.0),
        len(current_groups) / max(max_groups, 1),
        np.mean([g.sum() for g in current_groups]) / (current_groups[0].size if current_groups else 1.0),
    ], dtype=np.float32)


class ActionPolicy(nn.Module):
    def __init__(self, in_dim=7, hidden=32, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def collect_rl_episode(
    policy,
    fix_scorer,
    image_tensor,
    init_groups,
    candidate_bank,
    horizon=20,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    current = [g.copy() for g in init_groups]
    current_metrics = metrics_fix_only(fix_scorer, image_tensor, current)

    log_probs = []
    rewards = []
    trajectory = [dict(step=0, action="init", **current_metrics)]

    for step in range(1, horizon + 1):
        s = state_features(current, current_metrics, max_groups=max_groups)
        s_t = torch.from_numpy(s).unsqueeze(0).to(next(policy.parameters()).device)

        logits = policy(s_t)
        dist = torch.distributions.Categorical(logits=logits)
        action_idx = dist.sample()
        action_name = ACTION_NAMES[int(action_idx.item())]

        proposal = apply_action(current, action_name, candidate_bank, image01, rng, max_groups=max_groups)
        proposal_metrics = metrics_fix_only(fix_scorer, image_tensor, proposal)

        reward = proposal_metrics["fix"] - current_metrics["fix"]

        log_probs.append(dist.log_prob(action_idx))
        rewards.append(float(reward))

        current = [g.copy() for g in proposal]
        current_metrics = dict(proposal_metrics)

        trajectory.append(dict(step=step, action=action_name, reward=reward, **current_metrics))

    return log_probs, rewards, trajectory, current, current_metrics


def discounted_returns(rewards, gamma=0.95):
    out = []
    running = 0.0
    for r in rewards[::-1]:
        running = float(r) + gamma * running
        out.append(running)
    out = out[::-1]
    out = np.asarray(out, dtype=np.float32)
    if out.std() > 1e-8:
        out = (out - out.mean()) / (out.std() + 1e-8)
    return out

In [ ]:
def train_action_policy_reinforce(
    train_indices,
    dataset,
    fix_scorer,
    epochs=5,
    episodes_per_epoch=16,
    max_candidates=24,
    init_groups=3,
    horizon=15,
    lr=1e-3,
    gamma=0.95,
    seed=0,
):
    set_seed(seed)
    policy = ActionPolicy().to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)

    history = []

    for epoch in range(epochs):
        epoch_losses = []
        epoch_returns = []

        for _ in tqdm(range(episodes_per_epoch), desc=f"epoch {epoch+1}/{epochs}"):
            idx = int(np.random.choice(train_indices))
            item = dataset[idx]
            image_tensor = item["input"].unsqueeze(0).float()

            bank = build_candidate_bank_massmaps(
                image_tensor=image_tensor,
                max_candidates=max_candidates,
                min_area=20,
            )
            if len(bank["groups"]) == 0:
                continue

            singleton_df = ranked_singletons(fix_scorer, image_tensor, bank["groups"])
            start_groups = seed_init_groups(bank["groups"], singleton_df, init_groups=init_groups)

            log_probs, rewards, traj, _, _ = collect_rl_episode(
                policy=policy,
                fix_scorer=fix_scorer,
                image_tensor=image_tensor,
                init_groups=start_groups,
                candidate_bank=bank["groups"],
                horizon=horizon,
                seed=int(np.random.randint(0, 10_000_000)),
            )

            if len(log_probs) == 0:
                continue

            returns = discounted_returns(rewards, gamma=gamma)
            returns_t = torch.tensor(returns, device=device)
            log_probs_t = torch.stack(log_probs)

            loss = -(log_probs_t * returns_t).sum()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            epoch_losses.append(float(loss.item()))
            epoch_returns.append(float(np.sum(rewards)))

        summary = {
            "epoch": epoch + 1,
            "mean_loss": np.mean(epoch_losses) if epoch_losses else np.nan,
            "mean_episode_return": np.mean(epoch_returns) if epoch_returns else np.nan,
        }
        history.append(summary)
        print(summary)

    return policy, pd.DataFrame(history)

In [ ]:
def rollout_learned_policy(
    policy,
    fix_scorer,
    image_tensor,
    init_groups,
    candidate_bank,
    horizon=25,
    max_groups=8,
    seed=0,
):
    rng = np.random.default_rng(seed)
    image01 = to_numpy01(image_tensor[0, 0])

    current = [g.copy() for g in init_groups]
    current_metrics = metrics_fix_only(fix_scorer, image_tensor, current)

    best = [g.copy() for g in current]
    best_metrics = dict(current_metrics)
    history = [dict(step=0, action="init", **current_metrics)]

    for step in range(1, horizon + 1):
        with torch.no_grad():
            s = state_features(current, current_metrics, max_groups=max_groups)
            s_t = torch.from_numpy(s).unsqueeze(0).to(next(policy.parameters()).device)
            logits = policy(s_t)
            action_idx = torch.argmax(logits, dim=-1).item()
            action_name = ACTION_NAMES[int(action_idx)]

        proposal = apply_action(current, action_name, candidate_bank, image01, rng, max_groups=max_groups)
        proposal_metrics = metrics_fix_only(fix_scorer, image_tensor, proposal)

        current = [g.copy() for g in proposal]
        current_metrics = dict(proposal_metrics)

        if better_by_fix_then_div(current_metrics, best_metrics):
            best = [g.copy() for g in current]
            best_metrics = dict(current_metrics)

        history.append(dict(step=step, action=action_name, **current_metrics))

    return best, history, best_metrics

## Single-image walkthrough

In [ ]:
TEST_INDEX = 0
FIX_STEPS = 80
MAX_CANDIDATES = 32
INIT_GROUPS = 4
MAX_GROUPS = 8

set_seed(0)

item = test_dataset[TEST_INDEX]
image_tensor = item["input"].unsqueeze(0).float()
label = item["label"]

bank = build_candidate_bank_massmaps(
    image_tensor=image_tensor,
    max_candidates=MAX_CANDIDATES,
    min_area=20,
)

print("candidate count:", len(bank["groups"]))
singleton_df = ranked_singletons(massmaps_fix, image_tensor, bank["groups"])
display(singleton_df.head(10))

init_groups = seed_init_groups(bank["groups"], singleton_df, init_groups=INIT_GROUPS)
init_metrics = metrics_fix_only(massmaps_fix, image_tensor, init_groups)
print("initial metrics:", init_metrics)

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(bank["image01"], cmap="viridis")
plt.title("Mass map")
plt.axis("off")
plt.show()

plot_group_grid(bank["image01"], bank["groups"][:12], "Candidate bank (first 12)")
plot_group_grid(bank["image01"], init_groups, "Initial FIX-ranked groups")

In [ ]:
best_fix_groups, fix_history, best_fix_metrics = improve_groups_fix_only(
    fix_scorer=massmaps_fix,
    image_tensor=image_tensor,
    init_groups=init_groups,
    candidate_bank=bank["groups"],
    num_steps=FIX_STEPS,
    max_groups=MAX_GROUPS,
    seed=0,
)

print("best FIX-only metrics:", best_fix_metrics)
plot_group_grid(bank["image01"], best_fix_groups, "Best groups after FIX-only search")

In [ ]:
fix_df = pd.DataFrame(fix_history)
display(fix_df.head())

plt.figure(figsize=(8, 4))
plt.plot(fix_df["step"], fix_df["fix"], label="FIX")
plt.plot(fix_df["step"], fix_df["div"], label="diversity")
if "p_void_" in fix_df.columns:
    plt.plot(fix_df["step"], fix_df["p_void_"], label="p_void_")
if "p_cluster_" in fix_df.columns:
    plt.plot(fix_df["step"], fix_df["p_cluster_"], label="p_cluster_")
if "purity" in fix_df.columns:
    plt.plot(fix_df["step"], fix_df["purity"], label="purity")
plt.xlabel("step")
plt.ylabel("score")
plt.title("Mass Maps FIX-only search trajectory")
plt.legend()
plt.show()

print("FIX delta:", best_fix_metrics["fix"] - init_metrics["fix"])

## Random restarts and beam search

In [ ]:
restart_best_groups, restart_runs, restart_best_metrics, singleton_df = run_fix_search_with_restarts(
    fix_scorer=massmaps_fix,
    image_tensor=image_tensor,
    candidate_bank=bank["groups"],
    n_restarts=8,
    num_steps=60,
    init_groups=INIT_GROUPS,
    max_groups=MAX_GROUPS,
    seed=0,
)

restart_table = pd.DataFrame([
    {
        "restart": run["restart"],
        "fix": run["metrics"]["fix"],
        "div": run["metrics"]["div"],
        "p_void_": run["metrics"].get("p_void_", np.nan),
        "p_cluster_": run["metrics"].get("p_cluster_", np.nan),
        "purity": run["metrics"].get("purity", np.nan),
    }
    for run in restart_runs
]).sort_values(["fix", "div"], ascending=False)

display(restart_table)
print("best restart metrics:", restart_best_metrics)
plot_group_grid(bank["image01"], restart_best_groups, "Best groups after random restarts")

In [ ]:
beam_groups, beam_history, beam_metrics, beam_trace = beam_search_fix(
    fix_scorer=massmaps_fix,
    image_tensor=image_tensor,
    candidate_bank=bank["groups"],
    beam_width=4,
    expansions_per_state=4,
    num_rounds=10,
    max_groups=MAX_GROUPS,
    seed=0,
)

print("beam metrics:", beam_metrics)
print("beam trace:", beam_trace)
plot_group_grid(bank["image01"], beam_groups, "Best groups after beam search")

## Optional: train the RL-style action policy

In [ ]:
TRAIN_INDICES = list(range(min(64, len(train_dataset))))

policy, rl_train_df = train_action_policy_reinforce(
    train_indices=TRAIN_INDICES,
    dataset=train_dataset,
    fix_scorer=massmaps_fix,
    epochs=4,
    episodes_per_epoch=12,
    max_candidates=24,
    init_groups=3,
    horizon=12,
    lr=1e-3,
    gamma=0.95,
    seed=0,
)

display(rl_train_df)

In [ ]:
policy_groups, policy_history, policy_metrics = rollout_learned_policy(
    policy=policy,
    fix_scorer=massmaps_fix,
    image_tensor=image_tensor,
    init_groups=init_groups,
    candidate_bank=bank["groups"],
    horizon=20,
    max_groups=MAX_GROUPS,
    seed=0,
)

print("learned-policy metrics:", policy_metrics)
plot_group_grid(bank["image01"], policy_groups, "Groups from learned action policy")

## Small-batch evaluation

Use this to compare:
- initialization
- FIX-only search
- random restarts
- beam search
- learned policy rollout

In [ ]:
def evaluate_methods_on_subset(
    dataset,
    indices,
    fix_scorer,
    fix_steps=60,
    restarts=6,
    beam_rounds=8,
    max_candidates=32,
    init_groups=4,
    max_groups=8,
    policy=None,
):
    rows = []

    for idx in tqdm(indices):
        item = dataset[idx]
        image_tensor = item["input"].unsqueeze(0).float()

        bank = build_candidate_bank_massmaps(
            image_tensor=image_tensor,
            max_candidates=max_candidates,
            min_area=20,
        )
        if len(bank["groups"]) == 0:
            continue

        singleton_df = ranked_singletons(fix_scorer, image_tensor, bank["groups"])
        start_groups = seed_init_groups(bank["groups"], singleton_df, init_groups=init_groups)
        init_metrics = metrics_fix_only(fix_scorer, image_tensor, start_groups)

        _, _, fix_metrics = improve_groups_fix_only(
            fix_scorer=fix_scorer,
            image_tensor=image_tensor,
            init_groups=start_groups,
            candidate_bank=bank["groups"],
            num_steps=fix_steps,
            max_groups=max_groups,
            seed=idx,
        )

        _, _, restart_metrics, _ = run_fix_search_with_restarts(
            fix_scorer=fix_scorer,
            image_tensor=image_tensor,
            candidate_bank=bank["groups"],
            n_restarts=restarts,
            num_steps=max(20, fix_steps // 2),
            init_groups=init_groups,
            max_groups=max_groups,
            seed=idx,
        )

        _, _, beam_metrics, _ = beam_search_fix(
            fix_scorer=fix_scorer,
            image_tensor=image_tensor,
            candidate_bank=bank["groups"],
            beam_width=4,
            expansions_per_state=4,
            num_rounds=beam_rounds,
            max_groups=max_groups,
            seed=idx,
        )

        row = {
            "index": idx,
            "init_fix": init_metrics["fix"],
            "fix_search": fix_metrics["fix"],
            "restart_search": restart_metrics["fix"],
            "beam_search": beam_metrics["fix"],
        }

        if policy is not None:
            _, _, policy_metrics = rollout_learned_policy(
                policy=policy,
                fix_scorer=fix_scorer,
                image_tensor=image_tensor,
                init_groups=start_groups,
                candidate_bank=bank["groups"],
                horizon=20,
                max_groups=max_groups,
                seed=idx,
            )
            row["learned_policy"] = policy_metrics["fix"]

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
EVAL_INDICES = list(range(min(8, len(test_dataset))))

eval_df = evaluate_methods_on_subset(
    dataset=test_dataset,
    indices=EVAL_INDICES,
    fix_scorer=massmaps_fix,
    fix_steps=50,
    restarts=4,
    beam_rounds=6,
    max_candidates=32,
    init_groups=4,
    max_groups=8,
    policy=policy,
)

display(eval_df)

if len(eval_df):
    summary = pd.DataFrame({
        "mean": eval_df.mean(numeric_only=True),
        "std": eval_df.std(numeric_only=True),
    })
    display(summary)

## How to interpret the results

### If FIX-only search improves over initialization
Then **MassMapsFixScore can guide iterative feature improvement** in this setting.

### If restarts or beam help a lot
Then the main bottleneck is **search quality**, not lack of RL.

### If the learned policy beats plain search
Then there is real value in learning an action policy.

### If nothing improves much
Then the bottleneck is probably the **proposal bank**.

For mass maps, that usually means:
- better void/cluster proposal generators
- stronger intensity/topology priors
- multiscale threshold proposals
- morphology tuned to dark large voids vs bright small clusters

That is usually more important than adding more RL iterations.